In [29]:
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
import transformers

print(transformers.__version__)

4.57.1


In [30]:
ALPHABET = ["A", "B", "C", "D", "E", "F", "G"]
choice_template = "{choice}: {state}"
prompt_template = "{question} \n \n {choices} \n \n Answer with exactly one of the letters {letters} corresponding to the correct answer."

TASKS = {
    "drawer": {
        "question": "Is the wodden drawer under the table:",
        "classes": ["open", "closed"],
        "states": ["open", "closed"],
    },
    "light": {
        "question": "Is the rectangular shaped led light which emits green light:",
        "classes": ["on", "off"],
        "states": ["on", "off"],
    },
    "slider": {
        "question": "Is the slider door in the back of the table:",
        "classes": ["on the left", "on the right"],
        "states": ["left", "right"],
    },
    "red_block": {
        "question": "Is the red block:",
        "classes": ["on the table", "in the drawer", "not present"],
        "states": ["table", "drawer", "mia"],
    },
    "pink_block": {
        "question": "Is the pink block:",
        "classes": ["on the table", "in the drawer", "not present"],
        "states": ["table", "drawer", "mia"],
    },
    "blue_block": {
        "question": "Is the blue block:",
        "classes": ["on the table", "in the drawer", "not present"],
        "states": ["table", "drawer", "mia"],
    },
}


In [31]:
def get_choices(states:list[str]) -> str:
    return "\n".join([f"{choice}: {state}" for choice, state in zip(ALPHABET, states)])

def get_letters(num_classes:int) -> str:
    return ", ".join(ALPHABET[:num_classes])

def get_prompt(entity:str) -> str:
    question = TASKS[entity]["question"]
    states = TASKS[entity]["states"]
    choices = get_choices(states)
    letters = get_letters(len(states))
    return prompt_template.format(question=question, choices=choices, letters=letters)

In [32]:
prompts = []
images = []
ground_truth = []
task_names = []

# Build dataset
for entity, data in TASKS.items():
    prompt = get_prompt(entity)
    for state in data["states"]:
        for i in range(10):
            dir = "../data/samples/front"
            image = Image.open(f"{dir}/{entity}_{state}_{i}.png").convert("RGB")
            images.append(image)
            prompts.append(prompt)
            ground_truth.append(state)
            task_names.append(entity)

In [33]:
MODEL_NAME = "allenai/Molmo2-4B"
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.float32,
    device_map="auto",
)
model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Molmo2ForConditionalGeneration(
  (model): Molmo2Model(
    (transformer): Molmo2TextModel(
      (wte): Molmo2Embedding()
      (emb_drop): Dropout(p=0.0, inplace=False)
      (blocks): ModuleList(
        (0-35): 36 x Molmo2DecoderLayer(
          (self_attn): Molmo2Attention(
            (att_proj): Linear(in_features=2560, out_features=6144, bias=False)
            (k_norm): Molmo2RMSNorm((128,), eps=1e-06)
            (q_norm): Molmo2RMSNorm((128,), eps=1e-06)
            (attn_out): Linear(in_features=4096, out_features=2560, bias=False)
          )
          (attn_norm): Molmo2RMSNorm((2560,), eps=1e-06)
          (dropout): Dropout(p=0.0, inplace=False)
          (mlp): LanguageModelMLP(
            (ff_proj): Linear(in_features=2560, out_features=19456, bias=False)
            (ff_out): Linear(in_features=9728, out_features=2560, bias=False)
            (act): SiLUActivation()
          )
          (ff_norm): Molmo2RMSNorm((2560,), eps=1e-06)
        )
      )
      (ln_f): Mo

In [34]:
# Per-sample constrained decoding: only valid letters for that sample's task
cache_allowed_ids = {}
for entity, data in TASKS.items():
    ids = []
    for label in ALPHABET[: len(data["states"])]:
        token_ids = processor.tokenizer.encode(label, add_special_tokens=False)
        assert len(token_ids) == 1, f"Label {label} is not single-token for tokenizer"
        ids.append(token_ids[0])
    cache_allowed_ids[entity] = ids


def prefix_allowed_tokens_fn(batch_id, input_ids):
    return cache_allowed_ids[entity]

def postprocess_prediction(pred) -> str | None:
    pred = pred.strip().upper()
    if pred in ALPHABET:
        return pred
    else:
        return None
    

In [35]:
chat_texts = []
for prompt in prompts:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    chat_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    chat_texts.append(chat_text)

In [36]:
predicted_letters = []
img_batches = []
chat_batches = []
batch_size = 10
for i in range(0, len(images), batch_size):
    img_batches.append(images[i : i + batch_size])
    chat_batches.append(chat_texts[i : i + batch_size])

for img_batch, chat_batch in zip(img_batches, chat_batches):
    inputs = processor(
        images=img_batch,
        text=chat_batch,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    # Molmo2 + torch 2.5 workaround: avoid token_type_ids path that requires torch>=2.6.
    inputs.pop("token_type_ids", None)


    with torch.inference_mode():
        outputs = model.generate(  # type: ignore
            **inputs,
            max_new_tokens=1,
            do_sample=False,
            prefix_allowed_tokens_fn=prefix_allowed_tokens_fn,
        )

    input_lengths = inputs["attention_mask"].sum(dim=1)
    for i in range(outputs.shape[0]):
        gen_tokens = outputs[i, input_lengths[i] :]
        pred = processor.tokenizer.decode(gen_tokens, skip_special_tokens=True)
        post = postprocess_prediction(pred)
        predicted_letters.append(post)

In [37]:
# Evaluate
correct = 0
correct_per_task = {entity: 0 for entity in TASKS}
wrong_per_task = {entity: 0 for entity in TASKS}
for i in range(len(predicted_letters)):
    entity = task_names[i]
    states = TASKS[entity]["states"]
    letter_to_state = {ALPHABET[j]: state for j, state in enumerate(states)}
    predicted_state = letter_to_state.get(predicted_letters[i])

    is_ok = predicted_state == ground_truth[i]
    correct += int(is_ok)
    correct_per_task[entity] += int(is_ok)
    wrong_per_task[entity] += int(not is_ok)

    print(
        f"TASK={entity:<10} "
        f"GT={ground_truth[i]:<10} "
        f"PRED={predicted_state:<10} "
        f"LETTER={predicted_letters[i]}"
    )

acc = correct / max(len(predicted_letters), 1)
print(f"\nAccuracy: {acc:.3f} ({correct}/{len(predicted_letters)})")
for entity in TASKS:
    task_acc = correct_per_task[entity] / max(correct_per_task[entity] + wrong_per_task[entity], 1)
    print(f"TASK={entity:<10} Accuracy: {task_acc:.3f} ({correct_per_task[entity]}/{correct_per_task[entity] + wrong_per_task[entity]})")

TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=open       PRED=open       LETTER=A
TASK=drawer     GT=closed     PRED=closed     LETTER=B
TASK=drawer     GT=closed     PRED=closed     LETTER=B
TASK=drawer     GT=closed     PRED=closed     LETTER=B
TASK=drawer     GT=closed     PRED=closed     LETTER=B
TASK=drawer     GT=closed     PRED=closed     LETTER=B
TASK=drawer     GT=closed     PRED=closed     LETTER=B
TASK=drawer     GT=closed     PRED=closed     LETTER=B
TASK=drawer     GT=closed     PRED=closed     LETTER=B
TASK=drawe